# Contract Intelligence — LLM Serving on Colab (FastAPI + ngrok)

Self-hosts an **OpenAI-compatible** chat endpoint on a Colab GPU and exposes it publicly
via ngrok, so the DocIntel API (C2 RAG / C4 agent) can call it for grounded generation.

**All-in-one:** install → load model → FastAPI (`/v1/chat/completions`) → ngrok → wire into `.env`.

### Before you run
1. **Runtime → Change runtime type → GPU** (T4 works with 4-bit; L4/A100 for full precision).
2. Get a free ngrok authtoken: https://dashboard.ngrok.com/get-started/your-authtoken
3. Run cells top to bottom. The last setup cell prints the `DOCINTEL_LLM_BASE_URL` to paste
   into `docintel/.env` on your laptop.

The endpoint mirrors the OpenAI Chat Completions schema, so LangChain's `ChatOpenAI`
(used by `docintel.rag.llm.build_llm`) talks to it unchanged — only settings differ.


In [ ]:
# 1. Install dependencies (~2-3 min). torch/transformers preinstalled on Colab.
!pip install -q -U transformers accelerate bitsandbytes fastapi "uvicorn[standard]" pyngrok nest_asyncio


In [ ]:
# 2. Configuration — the only cell you normally edit.
import os

# Must match DOCINTEL_LLM_MODEL in docintel/.env
MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"

# ngrok authtoken (https://dashboard.ngrok.com/get-started/your-authtoken)
NGROK_AUTHTOKEN = "PASTE_YOUR_NGROK_AUTHTOKEN_HERE"

# Bearer token the server requires. Leave "EMPTY" to match a blank DOCINTEL_LLM_API_KEY.
API_KEY = "EMPTY"

# 4-bit (bitsandbytes) fits a 7B model on a 16 GB T4. Set False on L4/A100 for fp16.
LOAD_4BIT = True

# HuggingFace token — only needed for gated models (Qwen is open, leave blank).
HF_TOKEN = ""

PORT = 8000
MAX_NEW_TOKENS_CAP = 1024  # hard cap regardless of client request

if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
print("Config set. MODEL_ID =", MODEL_ID, "| 4-bit =", LOAD_4BIT)


In [ ]:
# 3. Load model + tokenizer onto the GPU.
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

assert torch.cuda.is_available(), "No GPU. Runtime -> Change runtime type -> GPU."
print("GPU:", torch.cuda.get_device_name(0))

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

load_kwargs = {"device_map": "auto", "torch_dtype": torch.bfloat16}
if LOAD_4BIT:
    from transformers import BitsAndBytesConfig

    load_kwargs["quantization_config"] = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )

model = AutoModelForCausalLM.from_pretrained(MODEL_ID, **load_kwargs)
model.eval()
print("Model loaded.")


In [ ]:
# 4. FastAPI app — OpenAI-compatible /v1/chat/completions (+ streaming), /v1/models, /health.
import time
import uuid
import json as _json
from threading import Thread
from typing import Any

from fastapi import FastAPI, HTTPException, Request
from fastapi.responses import StreamingResponse
from pydantic import BaseModel
from transformers import TextIteratorStreamer

app = FastAPI(title="DocIntel LLM (Colab)")


class ChatMessage(BaseModel):
    role: str
    content: str


class ChatCompletionRequest(BaseModel):
    model: str | None = None
    messages: list[ChatMessage]
    temperature: float = 0.0
    max_tokens: int | None = None
    stream: bool = False


def _check_auth(request: Request) -> None:
    if not API_KEY or API_KEY == "EMPTY":
        return
    header = request.headers.get("authorization", "")
    if header.removeprefix("Bearer ").strip() != API_KEY:
        raise HTTPException(status_code=401, detail="Invalid API key")


def _build_inputs(messages: list[ChatMessage]):
    chat = [{"role": m.role, "content": m.content} for m in messages]
    text = tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)
    return tokenizer(text, return_tensors="pt").to(model.device)


def _gen_kwargs(inputs, req: ChatCompletionRequest) -> dict[str, Any]:
    max_new = min(req.max_tokens or 512, MAX_NEW_TOKENS_CAP)
    kwargs: dict[str, Any] = {
        **inputs,
        "max_new_tokens": max_new,
        "pad_token_id": tokenizer.eos_token_id,
    }
    if req.temperature and req.temperature > 0:
        kwargs.update(do_sample=True, temperature=req.temperature)
    else:
        kwargs.update(do_sample=False)
    return kwargs


@app.get("/health")
def health() -> dict[str, Any]:
    return {"status": "ok", "model": MODEL_ID}


@app.get("/v1/models")
def list_models() -> dict[str, Any]:
    return {
        "object": "list",
        "data": [{"id": MODEL_ID, "object": "model", "created": int(time.time()), "owned_by": "local"}],
    }


@app.post("/v1/chat/completions")
def chat_completions(req: ChatCompletionRequest, request: Request):
    _check_auth(request)
    inputs = _build_inputs(req.messages)
    prompt_tokens = int(inputs["input_ids"].shape[1])
    created = int(time.time())
    cid = f"chatcmpl-{uuid.uuid4().hex}"

    if req.stream:
        streamer = TextIteratorStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)
        Thread(target=model.generate, kwargs={**_gen_kwargs(inputs, req), "streamer": streamer}).start()

        def event_stream():
            for piece in streamer:
                chunk = {
                    "id": cid, "object": "chat.completion.chunk", "created": created, "model": MODEL_ID,
                    "choices": [{"index": 0, "delta": {"content": piece}, "finish_reason": None}],
                }
                yield f"data: {_json.dumps(chunk)}\n\n"
            done = {
                "id": cid, "object": "chat.completion.chunk", "created": created, "model": MODEL_ID,
                "choices": [{"index": 0, "delta": {}, "finish_reason": "stop"}],
            }
            yield f"data: {_json.dumps(done)}\n\n"
            yield "data: [DONE]\n\n"

        return StreamingResponse(event_stream(), media_type="text/event-stream")

    with torch.no_grad():
        output = model.generate(**_gen_kwargs(inputs, req))
    gen_ids = output[0][prompt_tokens:]
    text = tokenizer.decode(gen_ids, skip_special_tokens=True).strip()
    completion_tokens = int(gen_ids.shape[0])

    return {
        "id": cid, "object": "chat.completion", "created": created, "model": MODEL_ID,
        "choices": [{"index": 0, "message": {"role": "assistant", "content": text}, "finish_reason": "stop"}],
        "usage": {
            "prompt_tokens": prompt_tokens,
            "completion_tokens": completion_tokens,
            "total_tokens": prompt_tokens + completion_tokens,
        },
    }


print("FastAPI app defined.")


In [ ]:
# 5. Launch uvicorn in a background thread and open the ngrok tunnel.
import nest_asyncio
import uvicorn
from threading import Thread
from pyngrok import ngrok, conf

nest_asyncio.apply()

# Reset any tunnels left over from a previous run in this session.
for t in ngrok.get_tunnels():
    ngrok.disconnect(t.public_url)

conf.get_default().auth_token = NGROK_AUTHTOKEN

def _serve():
    uvicorn.run(app, host="0.0.0.0", port=PORT, log_level="warning")

Thread(target=_serve, daemon=True).start()

public_url = ngrok.connect(PORT).public_url
base_url = f"{public_url}/v1"

print("Server is live.\n")
print("Paste these into docintel/.env on your laptop, then restart the API:\n")
print(f"DOCINTEL_LLM_BASE_URL={base_url}")
print(f"DOCINTEL_LLM_API_KEY={'' if API_KEY == 'EMPTY' else API_KEY}")
print(f"DOCINTEL_LLM_MODEL={MODEL_ID}")


In [ ]:
# 6. Self-test — round-trips the public endpoint exactly as DocIntel will.
import requests

resp = requests.post(
    f"{base_url}/chat/completions",
    headers={"Authorization": f"Bearer {API_KEY}"},
    json={
        "model": MODEL_ID,
        "messages": [
            {"role": "system", "content": "You are a contract analysis assistant."},
            {"role": "user", "content": "In one sentence, what is a governing law clause?"},
        ],
        "temperature": 0,
        "max_tokens": 128,
    },
    timeout=120,
)
resp.raise_for_status()
print(resp.json()["choices"][0]["message"]["content"])


## Wire it into DocIntel

1. Copy the three `DOCINTEL_LLM_*` lines printed in step 5 into `docintel/.env`.
2. Restart the DocIntel API so it re-reads settings.
3. `POST /contracts/ask` and the C4 agent now return **generated answers** (status
   `success` with citations) instead of the citations-only `degraded` response.

**Notes**
- Keep this Colab tab open — the tunnel dies when the runtime stops, and the ngrok URL
  changes every session (re-run step 5 and update `.env` each time).
- Free ngrok allows one tunnel; step 5 clears stale tunnels before opening a new one.
- To stop: `Runtime → Disconnect and delete runtime`, or run `ngrok.kill()` in a new cell.
